In [2]:
import urllib.request

# Configuration
dataset_name = "ruoka"
base_url = f"https://cs.uef.fi/~himat/WebRank/dataset_12/dataset_12/{dataset_name}"
num_pages = 100

# Extract GT keywords and store in a set
gt_keywords = set()

print(f"Extracting GT keywords from {dataset_name} dataset...")
for i in range(num_pages):
    try:
        gt_url = f"{base_url}/{i}/GT.txt"
        with urllib.request.urlopen(gt_url, timeout=5) as response:
            gt_text = response.read().decode("utf-8-sig").strip()
            keywords = gt_text.lower().split()
            gt_keywords.update(keywords)
    except:
        pass

print(f"✓ Extracted {len(gt_keywords)} unique GT keywords")
print(f"\nFirst 20 keywords: {list(gt_keywords)[:20]}")

Extracting GT keywords from ruoka dataset...
✓ Extracted 346 unique GT keywords

First 20 keywords: ['kala', 'kasvisruoat', 'jouluruoat', 'valmistujaiset', 'kuha', 'brûlée', 'maku', 'kananpoika', 'testaus', 'punajuuri', 'kanansiivet', 'laatikot', 'riisiruoka', 'varpula', 'curryt', 'riisipallot', 'kuohuviinit', 'parsakaali', 'paahtovanukas', 'liharuoat']


## Step 2: Apply Stemming

In [3]:
from nltk.stem.snowball import SnowballStemmer
import nltk

# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

# Initialize Finnish stemmer (ruoka dataset is Finnish)
stemmer = SnowballStemmer('finnish')

# Apply stemming
stemmed_keywords = {}
changed_count = 0

for keyword in gt_keywords:
    stemmed = stemmer.stem(keyword)
    stemmed_keywords[keyword] = stemmed
    if keyword != stemmed:
        changed_count += 1

# Show only changed keywords
print("STEMMING RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by stemming: {changed_count}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count}")
print(f"\nChanged keywords (Original → Stemmed):")
print("-" * 60)

changed_keywords = [(orig, stemmed) for orig, stemmed in stemmed_keywords.items() if orig != stemmed]
for i, (orig, stemmed) in enumerate(sorted(changed_keywords)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {stemmed}")

if len(changed_keywords) > 30:
    print(f"... and {len(changed_keywords) - 30} more changed keywords")

STEMMING RESULTS
Total keywords: 346
Keywords changed by stemming: 266
Keywords unchanged: 80

Changed keywords (Original → Stemmed):
------------------------------------------------------------
  1. aamiaine             → aamia
  2. aasialainen          → aasialain
  3. afrika               → afrik
  4. ankanrintaa          → ankanrint
  5. anna                 → an
  6. annosfondi           → annosfond
  7. appelsiinikastiketta → appelsiinikastik
  8. arancini             → aranc
  9. arkiruoat            → arkiruoa
 10. basmati              → basmat
 11. bataatti             → bataat
 12. broileri             → broiler
 13. broilerin            → broiler
 14. broilerinliha        → broilerinlih
 15. broileriruoat        → broileriruoa
 16. buffet               → buf
 17. carpaccio            → carpacio
 18. chicken              → chick
 19. couscoussalaatti     → couscoussalaat
 20. curry                → cury
 21. curryt               → cury
 22. espanja              → espanj
 23. 

## Step 3: Apply Lemmatization

In [4]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

# Download required NLTK data for lemmatization
for package in ['wordnet', 'omw-1.4', 'averaged_perceptron_tagger']:
    try:
        nltk.data.find(f'corpora/{package}')
    except LookupError:
        try:
            nltk.download(package, quiet=True)
        except:
            pass

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Apply lemmatization (using noun as default POS)
lemmatized_keywords = {}
changed_count_lemma = 0

for keyword in gt_keywords:
    # Try lemmatizing as noun, verb, and adjective, pick the shortest
    lemma_noun = lemmatizer.lemmatize(keyword, pos='n')
    lemma_verb = lemmatizer.lemmatize(keyword, pos='v')
    lemma_adj = lemmatizer.lemmatize(keyword, pos='a')
    
    # Pick the shortest lemma (most reduced form)
    lemmatized = min([lemma_noun, lemma_verb, lemma_adj], key=len)
    lemmatized_keywords[keyword] = lemmatized
    
    if keyword != lemmatized:
        changed_count_lemma += 1

# Show only changed keywords
print("LEMMATIZATION RESULTS")
print("=" * 60)
print(f"Total keywords: {len(gt_keywords)}")
print(f"Keywords changed by lemmatization: {changed_count_lemma}")
print(f"Keywords unchanged: {len(gt_keywords) - changed_count_lemma}")
print(f"\nChanged keywords (Original → Lemmatized):")
print("-" * 60)

changed_keywords_lemma = [(orig, lemma) for orig, lemma in lemmatized_keywords.items() if orig != lemma]
for i, (orig, lemma) in enumerate(sorted(changed_keywords_lemma)[:30], 1):
    print(f"{i:3d}. {orig:20s} → {lemma}")

if len(changed_keywords_lemma) > 30:
    print(f"... and {len(changed_keywords_lemma) - 30} more changed keywords")

LEMMATIZATION RESULTS
Total keywords: 346
Keywords changed by lemmatization: 1
Keywords unchanged: 345

Changed keywords (Original → Lemmatized):
------------------------------------------------------------
  1. tapas                → tapa


## Step 4: Compare Stemming vs Lemmatization

In [5]:
# Compare stemming vs lemmatization
print("COMPARISON: STEMMING vs LEMMATIZATION")
print("=" * 70)
print(f"{'Metric':<40s} {'Stemming':>12s} {'Lemmatization':>15s}")
print("-" * 70)
print(f"{'Total keywords':<40s} {len(gt_keywords):>12d} {len(gt_keywords):>15d}")
print(f"{'Keywords changed':<40s} {changed_count:>12d} {changed_count_lemma:>15d}")
print(f"{'Keywords unchanged':<40s} {len(gt_keywords)-changed_count:>12d} {len(gt_keywords)-changed_count_lemma:>15d}")
print(f"{'Change rate (%)':<40s} {changed_count/len(gt_keywords)*100:>11.1f}% {changed_count_lemma/len(gt_keywords)*100:>14.1f}%")
print("=" * 70)

# Find keywords where stemming and lemmatization differ
different_results = []
for keyword in gt_keywords:
    if stemmed_keywords[keyword] != lemmatized_keywords[keyword]:
        different_results.append((keyword, stemmed_keywords[keyword], lemmatized_keywords[keyword]))

print(f"\nKeywords with different results (Stemming ≠ Lemmatization): {len(different_results)}")
if different_results:
    print("\nExamples (Original → Stemmed → Lemmatized):")
    print("-" * 70)
    for i, (orig, stem, lemma) in enumerate(sorted(different_results)[:20], 1):
        print(f"{i:3d}. {orig:15s} → {stem:15s} → {lemma}")
    
    if len(different_results) > 20:
        print(f"... and {len(different_results) - 20} more differences")

COMPARISON: STEMMING vs LEMMATIZATION
Metric                                       Stemming   Lemmatization
----------------------------------------------------------------------
Total keywords                                    346             346
Keywords changed                                  266               1
Keywords unchanged                                 80             345
Change rate (%)                                 76.9%            0.3%

Keywords with different results (Stemming ≠ Lemmatization): 267

Examples (Original → Stemmed → Lemmatized):
----------------------------------------------------------------------
  1. aamiaine        → aamia           → aamiaine
  2. aasialainen     → aasialain       → aasialainen
  3. afrika          → afrik           → afrika
  4. ankanrintaa     → ankanrint       → ankanrintaa
  5. anna            → an              → anna
  6. annosfondi      → annosfond       → annosfondi
  7. appelsiinikastiketta → appelsiinikastik → appelsiinik

## Step 5: POS Analysis of Stemmed Keywords

In [6]:
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from collections import Counter

# Download required NLTK data for POS tagging
try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger', quiet=True)

# Get unique stemmed keywords
stemmed_unique = set(stemmed_keywords.values())
print(f"Total unique stemmed keywords: {len(stemmed_unique)}")
print(f"Original keywords: {len(gt_keywords)}")
print(f"Reduction from stemming: {len(gt_keywords) - len(stemmed_unique)} keywords merged\n")

# POS tag all stemmed keywords
pos_tagged = {}
pos_distribution = Counter()

print("Performing POS tagging on stemmed keywords...")
for stemmed_keyword in stemmed_unique:
    try:
        tokens = word_tokenize(stemmed_keyword)
        tags = pos_tag(tokens, tagset='universal')
        # Use the POS tag of the first token (most common in single-word keywords)
        pos = tags[0][1] if tags else 'UNK'
        pos_tagged[stemmed_keyword] = pos
        pos_distribution[pos] += 1
    except:
        pos_tagged[stemmed_keyword] = 'UNK'
        pos_distribution['UNK'] += 1

# POS mapping for readable labels
pos_labels = {
    'NOUN': 'Noun',
    'VERB': 'Verb',
    'ADJ': 'Adjective',
    'ADV': 'Adverb',
    'PRON': 'Pronoun',
    'ADP': 'Adposition',
    'CCONJ': 'Coord. Conjunction',
    'SCONJ': 'Subord. Conjunction',
    'DET': 'Determiner',
    'NUM': 'Number',
    'INTJ': 'Interjection',
    'PART': 'Particle',
    'PROPN': 'Proper Noun',
    'SYM': 'Symbol',
    'X': 'Other',
    'UNK': 'Unknown',
    '.': 'Punctuation'
}

print("\n" + "=" * 60)
print("POS DISTRIBUTION OF STEMMED KEYWORDS")
print("=" * 60)

# Sort by frequency
for pos, count in pos_distribution.most_common():
    label = pos_labels.get(pos, pos)
    percentage = (count / len(stemmed_unique)) * 100
    print(f"{label:20s}: {count:4d} ({percentage:6.2f}%)")

print("=" * 60)

# Show examples for each POS
print("\nEXAMPLES BY POS (first 5 of each):\n")

for pos in sorted(pos_distribution.keys(), key=lambda x: pos_distribution[x], reverse=True):
    keywords_with_pos = [k for k, p in pos_tagged.items() if p == pos]
    label = pos_labels.get(pos, pos)
    print(f"{label} ({len(keywords_with_pos)} keywords):")
    for keyword in sorted(keywords_with_pos)[:5]:
        print(f"  - {keyword}")
    print()

Total unique stemmed keywords: 332
Original keywords: 346
Reduction from stemming: 14 keywords merged

Performing POS tagging on stemmed keywords...

POS DISTRIBUTION OF STEMMED KEYWORDS
Noun                :  329 ( 99.10%)
Verb                :    2 (  0.60%)
Determiner          :    1 (  0.30%)

EXAMPLES BY POS (first 5 of each):

Noun (329 keywords):
  - aamia
  - aasialain
  - afrik
  - alkoholito
  - ankanrint

Verb (2 keywords):
  - can
  - kananuget

Determiner (1 keywords):
  - an



## Step 6: Character Length Analysis of Stemmed Keywords

In [7]:
import numpy as np

# Analyze character length of stemmed keywords
stemmed_list = list(stemmed_unique)
char_lengths = [len(keyword) for keyword in stemmed_list]

# Calculate statistics
length_stats = {
    'mean': np.mean(char_lengths),
    'median': np.median(char_lengths),
    'min': np.min(char_lengths),
    'max': np.max(char_lengths),
    'std': np.std(char_lengths),
    '25th_percentile': np.percentile(char_lengths, 25),
    '75th_percentile': np.percentile(char_lengths, 75)
}

print("\n" + "=" * 60)
print("CHARACTER LENGTH ANALYSIS - STEMMED KEYWORDS")
print("=" * 60)
print(f"Total unique stemmed keywords: {len(stemmed_list)}")
print(f"\nStatistics:")
print(f"  Mean length:         {length_stats['mean']:.2f} characters")
print(f"  Median length:       {length_stats['median']:.0f} characters")
print(f"  Min length:          {length_stats['min']} characters")
print(f"  Max length:          {length_stats['max']} characters")
print(f"  Std Dev:             {length_stats['std']:.2f}")
print(f"  25th percentile:     {length_stats['25th_percentile']:.0f} characters")
print(f"  75th percentile:     {length_stats['75th_percentile']:.0f} characters")

# Individual character length distribution
print(f"\nDetailed Length Distribution (by individual character count):")
print("-" * 60)
length_freq = {}
for length in char_lengths:
    length_freq[length] = length_freq.get(length, 0) + 1

for char_len in sorted(length_freq.keys()):
    count = length_freq[char_len]
    percentage = (count / len(char_lengths)) * 100
    bar_length = int(percentage / 2)  # Create a simple bar visualization
    bar = "█" * bar_length
    print(f"  {char_len:2d} characters: {count:4d} ({percentage:6.2f}%) {bar}")

# Summary by ranges
print(f"\nLength Distribution by Category:")
print("-" * 60)
length_ranges = [
    (1, 3, "Very Short (1-3)"),
    (4, 6, "Short (4-6)"),
    (7, 9, "Medium (7-9)"),
    (10, 12, "Long (10-12)"),
    (13, float('inf'), "Very Long (13+)")
]

for min_len, max_len, label in length_ranges:
    count = sum(1 for l in char_lengths if min_len <= l <= max_len)
    percentage = (count / len(char_lengths)) * 100
    print(f"  {label:20s}: {count:4d} ({percentage:6.2f}%)")

# Show examples by length range
print(f"\nExamples by Length Category:")
print("-" * 60)
for min_len, max_len, label in length_ranges:
    examples = [k for k in stemmed_list if min_len <= len(k) <= max_len]
    if examples:
        print(f"{label}:", ", ".join(sorted(examples)[:8]))
    print()

print("=" * 60)


CHARACTER LENGTH ANALYSIS - STEMMED KEYWORDS
Total unique stemmed keywords: 332

Statistics:
  Mean length:         7.59 characters
  Median length:       7 characters
  Min length:          2 characters
  Max length:          16 characters
  Std Dev:             2.98
  25th percentile:     5 characters
  75th percentile:     9 characters

Detailed Length Distribution (by individual character count):
------------------------------------------------------------
   2 characters:    3 (  0.90%) 
   3 characters:   12 (  3.61%) █
   4 characters:   38 ( 11.45%) █████
   5 characters:   48 ( 14.46%) ███████
   6 characters:   39 ( 11.75%) █████
   7 characters:   27 (  8.13%) ████
   8 characters:   38 ( 11.45%) █████
   9 characters:   44 ( 13.25%) ██████
  10 characters:   33 (  9.94%) ████
  11 characters:   14 (  4.22%) ██
  12 characters:   13 (  3.92%) █
  13 characters:    6 (  1.81%) 
  14 characters:   11 (  3.31%) █
  15 characters:    5 (  1.51%) 
  16 characters:    1 (  0.30%)

## Step 7: Scan Web Pages and Find GT Keywords (Sorted Ascending)

In [8]:
import pandas as pd
import re
from bs4 import BeautifulSoup

# Scan all web pages and find GT keywords
print("=" * 80)
print("STEP 7: SCANNING WEB PAGES FOR GT KEYWORDS (PER-SITE MATCHING)")
print("=" * 80)
print(f"\nScanning {num_pages} web pages for GT keyword matches...")
print("Each page is matched only against its own GT keywords (not global).\n")

page_records = []

for page_index in range(num_pages):
    try:
        # Load PAGE-SPECIFIC GT keywords
        gt_url = f"{base_url}/{page_index}/GT.txt"
        with urllib.request.urlopen(gt_url, timeout=5) as gt_response:
            gt_text = gt_response.read().decode("utf-8-sig").strip()
            page_gt_keywords = [k.lower() for k in gt_text.split()]
            page_gt_stems = set(stemmer.stem(keyword) for keyword in page_gt_keywords)
        
        # Fetch and parse HTML page
        page_url = f"{base_url}/{page_index}"
        with urllib.request.urlopen(page_url, timeout=8) as response:
            html = response.read()
        
        soup = BeautifulSoup(html, 'lxml')
        
        # Remove script, style, and noscript tags
        for tag in soup(['script', 'style', 'noscript']):
            tag.decompose()
        
        # Extract visible text
        text = soup.get_text()
        
        # Tokenize using Finnish character set
        tokens = re.findall(r"[a-zåäöA-ZÅÄÖ]+", text.lower())
        stemmed_tokens = [stemmer.stem(token) for token in tokens]
        
        # Match ONLY against page-specific GT stems
        matched_tokens = [tok for tok in stemmed_tokens if tok in page_gt_stems]
        unique_matched = sorted(set(matched_tokens))
        
        # Get page title
        title_tag = soup.find('title')
        title = title_tag.string if title_tag else "No title"
        
        # Calculate metrics
        gt_coverage_pct = (len(unique_matched) / len(page_gt_stems) * 100) if page_gt_stems else 0
        match_rate_pct = (len(matched_tokens) / len(tokens) * 100) if tokens else 0
        
        page_records.append({
            'page_index': page_index,
            'url': page_url,
            'title': title,
            'gt_keywords_count': len(page_gt_stems),
            'unique_matched_keywords': len(unique_matched),
            'matched_tokens_count': len(matched_tokens),
            'total_tokens': len(tokens),
            'gt_coverage_pct': gt_coverage_pct,
            'match_rate_pct': match_rate_pct,
            'matched_keyword_samples': ', '.join(unique_matched[:5])
        })
    except Exception as e:
        print(f"Error on page {page_index}: {str(e)[:100]}")

# Create DataFrame if we have records
if page_records:
    page_match_df = pd.DataFrame(page_records)
    
    # Sort from SMALLEST to MAXIMUM by gt_coverage_pct
    page_match_df_sorted = page_match_df.sort_values('gt_coverage_pct', ascending=True).reset_index(drop=True)
    
    print(f"\n✓ Successfully matched {len(page_records)} pages")
    print(f"\nAll {len(page_match_df_sorted)} pages sorted from SMALLEST to MAXIMUM GT Coverage %:\n")
    
    # Display all pages sorted ascending by gt_coverage_pct
    display_cols = ['page_index', 'gt_coverage_pct', 'unique_matched_keywords', 'gt_keywords_count', 'title']
    print(page_match_df_sorted[display_cols].to_string(index=False))
    
    # Summary statistics
    print(f"\n" + "=" * 80)
    print("SUMMARY STATISTICS")
    print("=" * 80)
    summary_stats = [
        ('Total pages scanned', len(page_match_df_sorted)),
        ('Min GT coverage % per page', f"{page_match_df_sorted['gt_coverage_pct'].min():.2f}%"),
        ('Max GT coverage % per page', f"{page_match_df_sorted['gt_coverage_pct'].max():.2f}%"),
        ('Mean GT coverage % per page', f"{page_match_df_sorted['gt_coverage_pct'].mean():.2f}%"),
        ('Median GT coverage % per page', f"{page_match_df_sorted['gt_coverage_pct'].median():.2f}%"),
    ]
    
    for label, value in summary_stats:
        print(f'{label:.<40s} {value}')
    
    print("=" * 80 + "\n")
else:
    print("No pages successfully scanned.")

STEP 7: SCANNING WEB PAGES FOR GT KEYWORDS (PER-SITE MATCHING)

Scanning 100 web pages for GT keyword matches...
Each page is matched only against its own GT keywords (not global).


✓ Successfully matched 100 pages

All 100 pages sorted from SMALLEST to MAXIMUM GT Coverage %:

 page_index  gt_coverage_pct  unique_matched_keywords  gt_keywords_count                                                                                         title
         62        71.428571                        5                  7                                                      \n   Inkivääri-limebrûlée - Ruoka.fi\n  
         59        83.333333                        5                  6                                                 \n   Horiátiki-maalaissalaatti - Ruoka.fi\n  
          9        85.714286                        6                  7         \n   Leivo italialaista leipää kotona - nappaa vinkit focaccian tekoon - Ruoka.fi\n  
         17        85.714286                        6

## Step 8: Stopword Classification

In [9]:
# Step 8: Stopword Classification using NLTK Finnish Stopwords
from nltk.corpus import stopwords

print("=" * 100)
print("STEP 8: STOPWORD CLASSIFICATION - NLTK FINNISH STOPWORDS")
print("=" * 100)

# Load Finnish stopwords
try:
    nltk_finnish_stopwords = set(stopwords.words('finnish'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    nltk_finnish_stopwords = set(stopwords.words('finnish'))

print(f"\nNLTK Finnish Stopwords Library: {len(nltk_finnish_stopwords)} stopwords")

# Get unique stemmed keywords
unique_stemmed = set(stemmed_keywords.values())

# Classify keywords
stopword_keywords = []
content_keywords = []

for keyword in unique_stemmed:
    if keyword in nltk_finnish_stopwords:
        stopword_keywords.append(keyword)
    else:
        content_keywords.append(keyword)

# Calculate statistics
total_keywords = len(unique_stemmed)
stopword_count = len(stopword_keywords)
content_count = len(content_keywords)
stopword_percentage = (stopword_count / total_keywords * 100) if total_keywords > 0 else 0
content_percentage = (content_count / total_keywords * 100) if total_keywords > 0 else 0

print(f"\nResults:")
print(f"  Total unique stemmed keywords: {total_keywords}")
print(f"  Stopwords: {stopword_count} ({stopword_percentage:.2f}%)")
print(f"  Content words: {content_count} ({content_percentage:.2f}%)")

if stopword_keywords:
    print(f"\n  Stopwords found: {', '.join(sorted(stopword_keywords))}")

print("\n" + "=" * 100)

STEP 8: STOPWORD CLASSIFICATION - NLTK FINNISH STOPWORDS

NLTK Finnish Stopwords Library: 229 stopwords

Results:
  Total unique stemmed keywords: 332
  Stopwords: 1 (0.30%)
  Content words: 331 (99.70%)

  Stopwords found: ja



## Step 9: GT Keywords in Website URLs

In [12]:
# Step 9: GT Keywords in URL Matching (Professional Pipeline with libvoikko)
from urllib.parse import urlparse, unquote
from libvoikko import Voikko

print("=" * 100)
print("STEP 9: GT KEYWORDS IN URLs (Professional: Compound Split -> Stem -> Filter -> Match)")
print("=" * 100)

# Define URL structural noise words
url_noise_words = {
    'resept', 'reseptit',
    'muka', 'mukana',
    'maku',
    'blogi', 'blogit',
    'ja',
    'helpo',
    'juhl', 'juhla',
    'pippurimyly'
}

print(f"Custom URL noise filter: {len(url_noise_words)} structural keywords")

# Initialize Finnish morphological analyzer
try:
    voikko_analyzer = Voikko('fi')
    voikko_enabled = True
    print("Compound splitter: libvoikko (enabled)")
except Exception:
    voikko_analyzer = None
    voikko_enabled = False
    print("Compound splitter: libvoikko unavailable, fallback mode enabled")

url_results = []
total_ulr_keywords_counter = 0
total_exact_match_counter = 0
total_fallback_match_counter = 0
total_compound_parts_generated = 0

for page_index in range(num_pages):
    try:
        # 1) Load URL and GT
        url_file_url = f"{base_url}/{page_index}/URL.txt"
        with urllib.request.urlopen(url_file_url, timeout=5) as url_response:
            real_url = url_response.read().decode("utf-8-sig").strip()

        gt_url = f"{base_url}/{page_index}/GT.txt"
        with urllib.request.urlopen(gt_url, timeout=5) as gt_response:
            gt_text = gt_response.read().decode("utf-8-sig").strip()
            page_gt_keywords = [k.lower() for k in gt_text.split()]

        # 2) GT stem+filter
        gt_stems = set(stemmer.stem(k) for k in page_gt_keywords)
        gt_content = gt_stems - nltk_finnish_stopwords

        # 3) URL normalize and tokenize
        parsed_url = urlparse(real_url)
        normalized_path = unquote(parsed_url.path.lower())
        raw_tokens = re.findall(r"[a-zåäöA-ZÅÄÖ0-9]+", normalized_path)

        # 4) Professional compound expansion with libvoikko
        expanded_tokens = []
        for token in raw_tokens:
            expanded_tokens.append(token)

            # split by simple boundaries first (already handled by regex over path separators/hyphens)
            if voikko_enabled and token.isalpha() and len(token) >= 5:
                analyses = voikko_analyzer.analyze(token)
                # print(f"analyses for '{token}': {analyses}")
                if analyses:
                    best = analyses[0]

                    # BASEFORM often useful as canonical token
                    baseform = best.get('BASEFORM')
                    if baseform:
                        expanded_tokens.append(baseform.lower())

                    # WORDBASES contains compound components, e.g. +nuudeli(nuudeli)+salaatti(salaatti)
                    wordbases = best.get('WORDBASES', '')
                    if wordbases:
                        compound_parts = re.findall(r"\(([^)]+)\)", wordbases)
                        for part in compound_parts:
                            cleaned_part = part.lower().strip()
                            if cleaned_part:
                                expanded_tokens.append(cleaned_part)
                                total_compound_parts_generated += 1

        # 5) Stem URL token universe
        url_stems = set(stemmer.stem(t) for t in expanded_tokens)

        # 6) Filter stopwords + URL noise
        url_content = url_stems - nltk_finnish_stopwords
        url_content = url_content - url_noise_words

        # 7) Matching
        exact_matches = url_content & gt_content

        # conservative fallback for unresolved compounds (only gt_stem contained in longer url stem)
        remaining_gt = gt_content - exact_matches
        remaining_url = url_content - exact_matches
        fallback_matches = set()

        for gt_stem in remaining_gt:
            if len(gt_stem) <= 3:
                continue
            for url_stem in remaining_url:
                if len(url_stem) > len(gt_stem) and gt_stem in url_stem:
                    fallback_matches.add(gt_stem)
                    break

        all_matches = exact_matches | fallback_matches

        # counters
        exact_match_count = len(exact_matches)
        fallback_match_count = len(fallback_matches)
        total_match_count = len(all_matches)

        total_exact_match_counter += exact_match_count
        total_fallback_match_counter += fallback_match_count
        total_ulr_keywords_counter += len(url_content)

        total_gt = len(gt_content)
        ratio = f"{total_match_count}/{len(url_content)}" if len(url_content) > 0 else "0/0"

        url_results.append({
            'page_index': page_index,
            'url': real_url,
            'matches': total_match_count,
            'exact_matches': exact_match_count,
            'fallback_matches': fallback_match_count,
            'total_gt_keywords': total_gt,
            'ratio': ratio,
            'matched_stems': ','.join(sorted(all_matches))
        })

    except Exception as e:
        print(f"Error on page {page_index}: {str(e)[:100]}")

# 8) Report
if url_results:
    results_df = pd.DataFrame(url_results).sort_values('matches', ascending=False).reset_index(drop=True)

    print(f"\nAnalyzed: {len(results_df)} pages\n")
    # print(results_df[['page_index', 'matches', 'exact_matches', 'fallback_matches', 'total_gt_keywords', 'ratio', 'url']].to_string(index=False))

    total_gt_match_counter = total_exact_match_counter + total_fallback_match_counter
    probability_pct = (total_gt_match_counter / total_ulr_keywords_counter * 100) if total_ulr_keywords_counter > 0 else 0.0

    print(f"\n  Total URL keywords (after filtering): {total_ulr_keywords_counter}")
    print(f"  Total GT exact matches: {total_exact_match_counter}")
    print(f"  Total GT fallback matches: {total_fallback_match_counter}")
    print(f"  Total GT matches: {total_gt_match_counter}")
    print(f"  Overall ratio: {total_gt_match_counter}/{total_ulr_keywords_counter if total_ulr_keywords_counter > 0 else 1}")
    print(f"  Probability P(GT in URL): {probability_pct:.2f}%")
    print(f"  Compound parts generated by libvoikko: {total_compound_parts_generated}")
    print(f"  Pipeline: URL normalize -> compound split -> stem -> stopword/noise filter -> match")
else:
    print("\n❌ No results collected. Check for errors above.")

if voikko_enabled and voikko_analyzer is not None:
    voikko_analyzer.terminate()

STEP 9: GT KEYWORDS IN URLs (Professional: Compound Split -> Stem -> Filter -> Match)
Custom URL noise filter: 12 structural keywords
Compound splitter: libvoikko (enabled)

Analyzed: 100 pages


  Total URL keywords (after filtering): 534
  Total GT exact matches: 169
  Total GT fallback matches: 28
  Total GT matches: 197
  Overall ratio: 197/534
  Probability P(GT in URL): 36.89%
  Compound parts generated by libvoikko: 462
  Pipeline: URL normalize -> compound split -> stem -> stopword/noise filter -> match


## Step 10: HTML Tag Analysis - GT Keyword Probability by Tag Type

In [13]:
# Step 10: Analyze GT Keywords in Various HTML Tags
print("=" * 100)
print("STEP 10: GT KEYWORDS IN HTML TAGS (Tag-by-Tag Analysis)")
print("=" * 100)
print(f"\nAnalyzing {num_pages} web pages for GT keyword matches across different HTML tags...")
print("Pipeline: Extract tag text -> tokenize -> stem -> filter -> match against page-specific GT\n")

# Define HTML tags to analyze
html_tags_to_analyze = [
    ('title', 'Title Tag', True),
    ('h1', 'H1 Headings', False),
    ('h2', 'H2 Headings', False),
    ('h3', 'H3 Headings', False),
    ('meta[name="description"]', 'Meta Description', True),
    ('meta[name="keywords"]', 'Meta Keywords', True),
    ('strong', 'Strong/Bold Text', False),
    ('em', 'Emphasis/Italic Text', False),
    ('a', 'Link Text (Anchor)', False),
    ('img[alt]', 'Image Alt Text', True),
]

# Storage for tag-specific results
tag_statistics = {}

for tag_selector, tag_name, is_attribute in html_tags_to_analyze:
    print(f"Processing: {tag_name}...")
    
    tag_token_counter = 0
    tag_match_counter = 0
    pages_with_tag = 0
    
    for page_index in range(num_pages):
        try:
            # Load page-specific GT keywords
            gt_url = f"{base_url}/{page_index}/GT.txt"
            with urllib.request.urlopen(gt_url, timeout=5) as gt_response:
                gt_text = gt_response.read().decode("utf-8-sig").strip()
                page_gt_keywords = [k.lower() for k in gt_text.split()]
            
            # GT stem + filter
            gt_stems = set(stemmer.stem(k) for k in page_gt_keywords)
            gt_content = gt_stems - nltk_finnish_stopwords
            
            # Fetch and parse HTML
            page_url = f"{base_url}/{page_index}"
            with urllib.request.urlopen(page_url, timeout=8) as response:
                html = response.read()
            
            soup = BeautifulSoup(html, 'lxml')
            
            # Extract text from specific tag type
            tag_text = ""
            if tag_selector == 'meta[name="description"]':
                meta_tag = soup.find('meta', attrs={'name': 'description'})
                if meta_tag and meta_tag.get('content'):
                    tag_text = meta_tag.get('content')
            elif tag_selector == 'meta[name="keywords"]':
                meta_tag = soup.find('meta', attrs={'name': 'keywords'})
                if meta_tag and meta_tag.get('content'):
                    tag_text = meta_tag.get('content')
            elif tag_selector == 'img[alt]':
                img_tags = soup.find_all('img', alt=True)
                tag_text = " ".join([img.get('alt', '') for img in img_tags])
            else:
                found_tags = soup.find_all(tag_selector)
                if found_tags:
                    tag_text = " ".join([tag.get_text() for tag in found_tags])
            
            if not tag_text.strip():
                continue
            
            pages_with_tag += 1
            
            # Tokenize using Finnish character set
            tokens = re.findall(r"[a-zåäöA-ZÅÄÖ]+", tag_text.lower())
            if not tokens:
                continue
            
            # Stem tokens
            stemmed_tokens = [stemmer.stem(token) for token in tokens]
            
            # Filter stopwords
            content_tokens = [tok for tok in stemmed_tokens if tok not in nltk_finnish_stopwords]
            
            # Match against page-specific GT
            matched_tokens = [tok for tok in content_tokens if tok in gt_content]
            
            tag_token_counter += len(content_tokens)
            tag_match_counter += len(matched_tokens)
            
        except Exception as e:
            continue
    
    # Calculate probability
    probability_pct = (tag_match_counter / tag_token_counter * 100) if tag_token_counter > 0 else 0.0
    
    tag_statistics[tag_name] = {
        'total_tokens': tag_token_counter,
        'matched_tokens': tag_match_counter,
        'probability_pct': probability_pct,
        'pages_with_tag': pages_with_tag,
        'avg_tokens_per_page': tag_token_counter / pages_with_tag if pages_with_tag > 0 else 0
    }

# Create summary DataFrame
summary_data = []
for tag_name, stats in tag_statistics.items():
    summary_data.append({
        'HTML Tag': tag_name,
        'Pages Found': stats['pages_with_tag'],
        'Total Tokens': stats['total_tokens'],
        'Matched GT': stats['matched_tokens'],
        'Probability P(GT)': f"{stats['probability_pct']:.2f}%",
        'Avg Tokens/Page': f"{stats['avg_tokens_per_page']:.1f}"
    })

summary_df = pd.DataFrame(summary_data)

# Sort by probability (descending)
summary_df['_prob_value'] = summary_df['Probability P(GT)'].str.rstrip('%').astype(float)
summary_df = summary_df.sort_values('_prob_value', ascending=False).drop('_prob_value', axis=1).reset_index(drop=True)

print("\n" + "=" * 100)
print("HTML TAG ANALYSIS RESULTS (Sorted by Probability - Highest to Lowest)")
print("=" * 100)
print(summary_df.to_string(index=False))

print("\n" + "=" * 100)
print("KEY INSIGHTS")
print("=" * 100)

# Find best and worst tags
best_tag_idx = summary_df.index[0]
worst_tag_idx = summary_df.index[-1]

best_tag = summary_df.iloc[best_tag_idx]['HTML Tag']
best_prob = summary_df.iloc[best_tag_idx]['Probability P(GT)']
worst_tag = summary_df.iloc[worst_tag_idx]['HTML Tag']
worst_prob = summary_df.iloc[worst_tag_idx]['Probability P(GT)']

print(f"🏆 Best Tag:  {best_tag:30s} - {best_prob} of tokens match GT keywords")
print(f"📉 Worst Tag: {worst_tag:30s} - {worst_prob} of tokens match GT keywords")

# Recommended weights for ranking system
print(f"\n📊 RECOMMENDED WEIGHTS FOR KEYWORD EXTRACTION RANKING SYSTEM:")
print("-" * 100)

total_prob = sum([float(row['Probability P(GT)'].rstrip('%')) for _, row in summary_df.iterrows()])
for idx, row in summary_df.head(6).iterrows():
    tag = row['HTML Tag']
    prob = float(row['Probability P(GT)'].rstrip('%'))
    recommended_weight = (prob / total_prob * 100) if total_prob > 0 else 0
    print(f"  {tag:30s}: {recommended_weight:5.1f}% weight  (P(GT) = {row['Probability P(GT)']}, {row['Pages Found']} pages)")

print("\n" + "=" * 100)

STEP 10: GT KEYWORDS IN HTML TAGS (Tag-by-Tag Analysis)

Analyzing 100 web pages for GT keyword matches across different HTML tags...
Pipeline: Extract tag text -> tokenize -> stem -> filter -> match against page-specific GT

Processing: Title Tag...
Processing: H1 Headings...
Processing: H2 Headings...
Processing: H3 Headings...
Processing: Meta Description...
Processing: Meta Keywords...
Processing: Strong/Bold Text...
Processing: Emphasis/Italic Text...
Processing: Link Text (Anchor)...
Processing: Image Alt Text...

HTML TAG ANALYSIS RESULTS (Sorted by Probability - Highest to Lowest)
            HTML Tag  Pages Found  Total Tokens  Matched GT Probability P(GT) Avg Tokens/Page
         H1 Headings          100           275         121            44.00%             2.8
           Title Tag          100           473         194            41.01%             4.7
    Strong/Bold Text           41           296          59            19.93%             7.2
      Image Alt Text        